# Day 22 — Advanced pandas: apply vs vectorization, query/eval, categoricals
Objectives:
- Replace apply with vectorized ops where possible.
- Use query/eval for performance.
- Use Categorical dtype to reduce memory.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-22`. Read
`python/ds-60day/companion-guides/day22_advanced_pandas_apply_query_eval.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Pandas is fastest and clearest when an operation can be expressed over
entire columns: arithmetic, comparisons, string/datetime accessors,
mapping, masks, or group transforms. A row-wise `apply(axis=1)` builds a
Series and calls Python for each row, so it should be a last resort for
genuinely row-dependent logic, not a default.

`query` and `eval` provide readable expression syntax but introduce
another name-resolution layer. Categoricals store repeated labels as
integer codes plus a level table; they can reduce memory and encode a
closed vocabulary, but nearly unique values may use more memory. Measure
before and after and define behavior for unseen categories.

### Vocabulary

- **vectorization:** column/array operations executed without a Python call per row.
- **row-wise apply:** calling a Python function with each row Series.
- **expression:** a calculation/filter written for `query` or `eval`.
- **categorical:** codes plus a finite table of allowed label values.
- **cardinality:** the count of distinct values in a column.
- **memory profiling:** measuring retained memory under a stated representation.

## Syntax anatomy

`frame.query("amount > @threshold")` resolves `amount` as a column and
`@threshold` from Python scope. `frame.eval("rate = part / whole")` can
assign an expression result. `series.astype("category")` creates codes
and categories; `memory_usage(deep=True)` includes referenced string
storage for a more honest comparison.

### Worked example 1 — Replace row-wise conditional logic with masks

Column expressions state each rule and preserve index alignment. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import numpy as np
import pandas as pd

frame = pd.DataFrame({"amount": [5, 20, 60]})
frame["band"] = np.select(
    [frame["amount"].ge(50), frame["amount"].ge(10)],
    ["high", "medium"],
    default="low",
)
frame.to_dict("records")

**Expected observation:** `[{'amount': 5, 'band': 'low'}, {'amount': 20, 'band': 'medium'}, {'amount': 60, 'band': 'high'}]`.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Measure categorical conversion

Keep the conversion only when data characteristics justify it. Predict first; then run the next cell.

In [ ]:
labels = pd.Series(["east", "west"] * 1_000, name="region")
categorical = labels.astype("category")
{
    "unique": labels.nunique(),
    "rows": len(labels),
    "object_bytes": int(labels.memory_usage(deep=True)),
    "category_bytes": int(categorical.memory_usage(deep=True)),
}

**Expected observation:** Two unique labels across 2,000 rows are reported, and the categorical representation is normally smaller. Exact byte counts vary by pandas version.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Classify row-wise logic as arithmetic, condition, string, mapping, or group transform before accepting `apply`.
2. List conditions from most specific to fallback and test missing values separately.
3. In `query`, distinguish column names from `@` external variables.
4. Measure categorical memory deeply and review category alignment before concatenation.

**Alternative to compare:** A named Python function plus `apply` is acceptable for truly irregular per-row objects; benchmark and keep the contract explicit.

**Boundary to test:** Missing values in conditions, divide-by-zero, nearly unique strings, unseen category levels, and unsafe dynamic expressions require care.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns
df = sns.load_dataset('tips')
# Vectorized tip %
df['tip_pct'] = df['tip']/df['total_bill']
# Categoricals
df['day'] = df['day'].astype('category')
df.dtypes
# query/eval
df.query('tip_pct > 0.2 and size >= 3').head()


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Find one row-wise `apply(axis=1)` in a supplied example and replace it with column arithmetic, string methods, mapping, masks, `np.select`, or group transform as appropriate.
   **Expected behavior:** values and index match the original for normal and missing inputs. **Constraints:** do not optimize by changing the contract.
   **Verify:** use `pd.testing` for equality and measure repeated execution on representative data.

2. Convert a repeated string column to categorical only after profiling. **Evidence:** record row count, unique count, object memory, categorical memory, and the category levels.
   **Expected behavior:** keep the categorical version only if it reduces memory for this data.
   **Verify:** values remain equivalent and explain why a nearly unique ID may become larger.

### Additional mastery practice

Prefer vectorized operations and labeled expressions, but measure rather than assuming. Use categoricals only when repetition justifies them.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict why row-wise `apply(axis=1)` is usually slower than column arithmetic for a simple ratio.
   **Progressive hint:** Vectorized operations avoid constructing/calling Python work per row.
   **Verify:** Assert row-wise and vectorized ratios agree including missing values, then report repeated timings on the same frame rather than a single anecdote.
4. **Tracing:** Trace `frame.query('amount > @threshold')`: which name comes from a column and which comes from Python scope?
   **Progressive hint:** `@` marks an external Python variable.
   **Verify:** Set a known threshold and assert the selected row indexes; change only the Python variable and confirm `@threshold`—not a column—controls the result.
5. **Implementation:** Write a function that compares memory before/after categorical conversion and keeps the category only when it reduces memory.
   **Progressive hint:** Use `memory_usage(deep=True)` on the Series.
   **Verify:** Assert values are unchanged, record both deep-memory counts, and return categorical only in the fixture where its bytes are smaller.
6. **Debugging:** Replace a row-wise conditional `apply` with `np.select` or `.where` while preserving missing-value behavior.
   **Progressive hint:** List conditions from most specific to fallback.
   **Verify:** Use rows covering every condition plus missing input; assert vectorized labels match the reference apply and missing policy exactly.
7. **Edge case and explanation:** Explain why a nearly unique string ID can consume more memory as a category and why category levels must be handled during concatenation.
   **Progressive hint:** Categories store both codes and a level table.
   **Verify:** Profile a repeated label and a nearly unique ID; assert the measured memory directions and reconcile category levels before/after concatenation.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Find one row-wise `apply(axis=1)` in a supplied example and replace it with column arithmetic, string methods, mapping, masks, `np.select`, or group transform as appropriate. **Expected behavior:** values and index match the original for normal and missing inputs. **Constraints:** do not optimize by changing the contract. **Verify:** use `pd.testing` for equality and measure repeated execution on representative data.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Find one row-wise `apply(axis=1)` in a supplied example and replace it with column arithmetic, string methods, mapping, masks, `np.select`, or group transform as appropriate. va...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Convert a repeated string column to categorical only after profiling. **Evidence:** record row count, unique count, object memory, categorical memory, and the category levels. **Expected behavior:** keep the categorical version only if it reduces memory for this data. **Verify:** values remain equivalent and explain why a nearly unique ID may become larger.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Convert a repeated string column to categorical only after profiling. record row count, unique count, object memory, categorical memory, and the category levels. keep the catego...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict why row-wise `apply(axis=1)` is usually slower than column arithmetic for a simple ratio. **Progressive hint:** Vectorized operations avoid constructing/calling Python work per row. **Verify:** Assert row-wise and vectorized ratios agree including missing values, then report repeated timings on the same frame rather than a single anecdote.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict why row-wise `apply(axis=1)` is usually slower than column arithmetic for a simple ratio. Vectorized operations avoid constructing/calling Python work per row. Assert ro...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace `frame.query('amount > @threshold')`: which name comes from a column and which comes from Python scope? **Progressive hint:** `@` marks an external Python variable. **Verify:** Set a known threshold and assert the selected row indexes; change only the Python variable and confirm `@threshold`—not a column—controls the result.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace `frame.query('amount > @threshold')`: which name comes from a column and which comes from Python scope? `@` marks an external Python variable. Set a known threshold and as...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Write a function that compares memory before/after categorical conversion and keeps the category only when it reduces memory. **Progressive hint:** Use `memory_usage(deep=True)` on the Series. **Verify:** Assert values are unchanged, record both deep-memory counts, and return categorical only in the fixture where its bytes are smaller.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Write a function that compares memory before/after categorical conversion and keeps the category only when it reduces memory. Use `memory_usage(deep=True)` on the Series. Assert...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Replace a row-wise conditional `apply` with `np.select` or `.where` while preserving missing-value behavior. **Progressive hint:** List conditions from most specific to fallback. **Verify:** Use rows covering every condition plus missing input; assert vectorized labels match the reference apply and missing policy exactly.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Replace a row-wise conditional `apply` with `np.select` or `.where` while preserving missing-value behavior. List conditions from most specific to fallback. Use rows covering ev...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Explain why a nearly unique string ID can consume more memory as a category and why category levels must be handled during concatenation. **Progressive hint:** Categories store both codes and a level table. **Verify:** Profile a repeated label and a nearly unique ID; assert the measured memory directions and reconcile category levels before/after concatenation.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Explain why a nearly unique string ID can consume more memory as a category and why category levels must be handled during concatenation. Categories store both codes and a level...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
